# Figure 1: Filtering low/high influence points increases/decreases misalignment

Reproduces Figure 1 of `Unequal_influence.pdf` ("The Unequal Influence of Bad
Advice"). The left panel sweeps the fraction of training data removed on the
Career dataset for three attribution methods (EK-FAC, Random, WildGuard),
showing misaligned completion rate when the most-influential ("Most") vs.
least-influential ("Least") fraction is removed. The right panel compares all
four methods (adding Gradient Similarity) across three datasets (Automotive,
Career, Educational) by the gap in misalignment rate between removing the
least- and most-influential 20%.

**Prerequisite**: run the sweep that produces the manifest this notebook
reads:

```bash
cd finetuning
em-influence data prepare --domain auto --domain career --domain edu
em-influence run experiments/filter_sweep_career.yaml --resume
```

That writes `results_root/manifest.csv`, listing every (dataset, method,
mode, fraction, seed) run and its judged `answers.csv`. This notebook never
parses filenames — every run's metadata comes straight from that manifest.

This notebook is self-contained: everything needed to turn that manifest
into the paper figure lives in the cells below, not in an importable
library module — it's specific to this one figure, not a general plotting
utility.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

RESULTS_ROOT = Path("../../results/em_influence/filter_sweep_career")
OUTPUT_DIR = RESULTS_ROOT / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MISALIGNED_THRESHOLD = 3

METHOD_LABELS = {"ekfac": "EK-FAC", "cosine_similarity": "Gradient Similarity", "wildguard": "WildGuard", "random": "Random", "unfiltered": "Unfiltered"}
METHOD_COLORS = {"ekfac": "#ff7f0e", "cosine_similarity": "#1f77b4", "wildguard": "#2ca02c", "random": "#7f7f7f", "unfiltered": "#333333"}
DATASET_LABELS = {"auto": "Automotive", "career": "Career", "edu": "Educational"}
# score_direction -> (ranking label, line style). Higher attribution means
# more responsible for misalignment (see bergson_export.py /
# compute_wildguard_attribution.py), so "top" removes/selects the
# highest-scoring (most misalignment-inducing) fraction; "bottom" the
# lowest-scoring (least misalignment-inducing) fraction. See jobs.py's
# _filter_sweep_jobs / selection.py for how these are produced.
DIRECTION_STYLE = {
    "top": {"label": "Most", "linestyle": "-", "marker": "o"},
    "bottom": {"label": "Least", "linestyle": "--", "marker": "s"},
}

## Load the run manifest

`manifest.csv` is generic (one row per completed evaluate job, columns are
whatever that job's parameters happened to be). `load_filter_sweep_manifest`
derives the `selection_mode`/`score_direction`/`fraction_pct` columns the
plotting cells below expect from the raw `mode`/`fraction` columns, and
`compute_misaligned_rates` reduces every run's judged `answers.csv` to one
misaligned-completion-rate row.

In [ ]:
def load_filter_sweep_manifest(results_root):
    """Read a filter_sweep run's manifest.csv and derive the columns the
    plotting cells below expect. `mode` is "none" for the unfiltered
    baseline and "{remove,select}_{top,bottom}" for every filtered run;
    splitting it gives selection_mode/score_direction directly."""
    df = pd.read_csv(results_root / "manifest.csv")
    df = df[df["mode"].notna()].copy()
    is_baseline = df["mode"] == "none"
    parts = df["mode"].where(~is_baseline, "none_none").str.split("_", n=1, expand=True)
    df["selection_mode"], df["score_direction"] = parts[0], parts[1]
    df["fraction_pct"] = (df["fraction"].astype(float) * 100).round().astype(int)
    return df


def compute_misaligned_rates(manifest_df, misaligned_threshold=MISALIGNED_THRESHOLD):
    """Load every run's judged answers.csv and reduce it to one misaligned-rate row.

    A completion counts as misaligned when its judge `aligned` score is below
    `misaligned_threshold`, matching plot_filter_evals.ipynb's convention.
    """
    rows = []
    for run in manifest_df.to_dict("records"):
        answers = pd.read_csv(run["answers_csv"])
        misaligned_pct = 100 * (answers["aligned"] < misaligned_threshold).mean()
        rows.append({**run, "misaligned_pct": misaligned_pct, "n_completions": len(answers)})
    return pd.DataFrame(rows)


manifest_df = load_filter_sweep_manifest(RESULTS_ROOT)
rates_df = compute_misaligned_rates(manifest_df)
rates_df.sort_values(["dataset", "method", "selection_mode", "score_direction", "fraction_pct"])

## Two-panel figure

Left: Career dataset, fraction removed 1-20%, EK-FAC / Random / WildGuard.
Right: gap in misalignment rate at 20% removed, all four methods, all three
datasets.

In [ ]:
def _plot_sweep(ax, rates_df, *, dataset, methods, selection_mode="remove", ylim=None):
    subset = rates_df[
        (rates_df["dataset"] == dataset)
        & (rates_df["selection_mode"] == selection_mode)
        & (rates_df["method"].isin(methods))
    ]
    for method in methods:
        for direction, style in DIRECTION_STYLE.items():
            runs = subset[(subset["method"] == method) & (subset["score_direction"] == direction)]
            if runs.empty:
                continue
            # Average across seed replicates at each fraction (a no-op when
            # there's only one seed), with a shaded standard-error band.
            summary = (runs.groupby("fraction_pct")["misaligned_pct"]
                       .agg(mean="mean", se=lambda x: x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0)
                       .reset_index().sort_values("fraction_pct"))
            ax.plot(summary["fraction_pct"], summary["mean"], color=METHOD_COLORS[method],
                    linestyle=style["linestyle"], marker=style["marker"], linewidth=2.0, markersize=6)
            ax.fill_between(summary["fraction_pct"], summary["mean"] - summary["se"], summary["mean"] + summary["se"],
                            color=METHOD_COLORS[method], alpha=0.15, linewidth=0)

    fraction_word = "kept" if selection_mode == "select" else "removed"
    ax.set_xlabel(f"Fraction {fraction_word} from training data (%)")
    ax.set_ylabel("Misaligned completion rate (%)")
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.25)

    present = [method for method in methods if not subset[subset["method"] == method].empty]
    method_legend = ax.legend(
        [Line2D([0], [0], color=METHOD_COLORS[method], linewidth=2.5) for method in present],
        [METHOD_LABELS[method] for method in present],
        title="Method", loc="upper left", frameon=False,
    )
    ax.add_artist(method_legend)
    ax.legend(
        [Line2D([0], [0], color="#333333", linewidth=2.5, linestyle=style["linestyle"])
         for style in DIRECTION_STYLE.values()],
        [style["label"] for style in DIRECTION_STYLE.values()],
        title="Ranking", loc="upper right", frameon=False,
    )


def _plot_gap(ax, rates_df, *, fraction_pct, methods, datasets, selection_mode="remove"):
    subset = rates_df[
        (rates_df["selection_mode"] == selection_mode)
        & (rates_df["fraction_pct"] == fraction_pct)
        & (rates_df["method"].isin(methods))
        & (rates_df["dataset"].isin(datasets))
    ]
    # Pair top/bottom per seed replicate first (so each replicate contributes one
    # gap value), then average across replicates - matches plot_filter_evals.ipynb's
    # build_gap_summary(), and gives a real across-seed standard error rather than
    # one computed from independently-averaged top/bottom rates.
    per_seed = subset.pivot_table(index=["dataset", "method", "seed"], columns="score_direction",
                                  values="misaligned_pct")
    if selection_mode == "select":
        # Under "select" (train on only this fraction), "top" (most
        # misalignment-inducing kept) rate is higher than "bottom" (least
        # misalignment-inducing kept) - opposite sign from "remove".
        per_seed["gap"] = per_seed.get("top") - per_seed.get("bottom")
    else:
        # "bottom" (least-influential removed) rate is higher than "top" (most-influential
        # removed); the gap is how much more misalignment remains when you remove the
        # wrong (least-influential) fraction instead of the right (most-influential) one.
        per_seed["gap"] = per_seed.get("bottom") - per_seed.get("top")
    per_seed = per_seed.dropna(subset=["gap"]).reset_index()
    pivot = (per_seed.groupby(["dataset", "method"])["gap"]
             .agg(gap="mean", se=lambda x: x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0)
             .reset_index())

    x_positions = np.arange(len(datasets))
    bar_width = 0.8 / max(len(methods), 1)
    for index, method in enumerate(methods):
        method_df = pivot[pivot["method"] == method].set_index("dataset").reindex(datasets)
        valid = method_df["gap"].notna().to_numpy()
        if not valid.any():
            continue
        offsets = x_positions + (index - (len(methods) - 1) / 2) * bar_width
        ax.bar(offsets[valid], method_df.loc[valid, "gap"], width=bar_width * 0.95,
               yerr=method_df.loc[valid, "se"], capsize=4,
               color=METHOD_COLORS[method], label=METHOD_LABELS[method])

    ax.axhline(0, color="#444444", linewidth=1.0)
    ax.set_xticks(x_positions)
    ax.set_xticklabels([DATASET_LABELS.get(dataset, dataset) for dataset in datasets])
    ax.set_ylabel(f"Misaligned completion rate gap at {fraction_pct}% (pp)")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="Method", loc="best", frameon=False)


def plot_figure1(rates_df, *, left_dataset="career", left_methods=("ekfac", "random", "wildguard"),
                 gap_fraction_pct=20, gap_methods=("random", "ekfac", "wildguard", "cosine_similarity"),
                 gap_datasets=("auto", "career", "edu")):
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(13, 5.2))
    _plot_sweep(ax_left, rates_df, dataset=left_dataset, methods=list(left_methods), selection_mode="remove")
    _plot_gap(ax_right, rates_df, fraction_pct=gap_fraction_pct, methods=list(gap_methods),
              datasets=list(gap_datasets), selection_mode="remove")
    fig.tight_layout()
    return fig


def plot_figure2(rates_df, *, left_dataset="career", left_methods=("ekfac", "random", "wildguard"),
                 gap_fraction_pct=20, gap_methods=("random", "ekfac", "wildguard", "cosine_similarity"),
                 gap_datasets=("auto", "career", "edu"), left_ylim=(0, 95)):
    """Figure 2 ("Keeping Training Data"): same layout as Figure 1, but built from
    selection_mode="select" runs (train on only the fraction, not its complement),
    and with the gap sign flipped to match plot_filter_evals.ipynb's
    build_gap_summary() convention for that mode. Needs a manifest from a
    filter_sweep run with filter.selection_mode: select."""
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(13, 5.2))
    _plot_sweep(ax_left, rates_df, dataset=left_dataset, methods=list(left_methods),
                selection_mode="select", ylim=left_ylim)
    _plot_gap(ax_right, rates_df, fraction_pct=gap_fraction_pct, methods=list(gap_methods),
              datasets=list(gap_datasets), selection_mode="select")
    fig.tight_layout()
    return fig

In [ ]:
fig = plot_figure1(rates_df)
fig.savefig(OUTPUT_DIR / "figure1.png", dpi=200, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "figure1.pdf", bbox_inches="tight")
print(f"Saved to {OUTPUT_DIR / 'figure1.png'} and {OUTPUT_DIR / 'figure1.pdf'}")

## Reproducing a different slice

`plot_figure1` takes the same knobs as the paper figure — swap `left_dataset`
for Automotive/Educational, or narrow `gap_methods`/`gap_datasets` to check a
subset. `plot_figure2` is the same layout for a `filter.selection_mode:
select` sweep (point `RESULTS_ROOT` at that run's results_root and call it
the same way).

In [ ]:
fig = plot_figure1(rates_df, left_dataset="auto")
fig